# Pengerukan pasir laut: suara publik dan aturan berpihak ke mana?

Indonesia melarang ekspor pasir laut sejak 2003, membukanya lagi lewat PP 26/2023,
lalu dilarang kembali oleh putusan Mahkamah Agung pada 2025. Aturannya berayun
seperti bandul. Di tengah ayunan itu, bagaimana nada pemberitaan media?
Saya mendapatkan 1.371 judul berita bertopik pasir laut dari Januari 2022 sampai
September 2026, menyaring 735 judul yang benar-benar membahas topik ini, lalu
mengukur proporsi judul bernada menolak vs mendukung di tiap babak kebijakan.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

from tools import gaya
from statsmodels.stats.proportion import proportion_confint, confint_proportions_2indep

DATA = Path("data")
df = pd.read_csv(DATA / "berita-berlabel.csv", parse_dates=["tanggal"])
linimasa = pd.read_csv(DATA / "linimasa-regulasi.csv", parse_dates=["tanggal"])
sub = df[df["relevan"]].copy()
print(f"Total Berita: {len(df)} judul | relevan: {len(sub)} | media berbeda: {sub['sumber'].nunique()}")
print(f"Rentang: {sub['tanggal'].min().date()} s.d. {sub['tanggal'].max().date()}")
print(sub["episode"].value_counts().sort_index().to_string())

Total Berita: 1371 judul | relevan: 735 | media berbeda: 246
Rentang: 2022-01-12 s.d. 2026-09-15
episode
E0_sebelum_pp           34
E1_pp_disahkan         259
E2_aturan_persiapan     59
E3_ekspor_dibuka       248
E4_pasca_putusan_ma    135


## Pokok cerita: lima babak regulasi

Linimasa dikurasi dari pemberitaan dan dokumen peraturan (daftar lengkap beserta
rujukannya ada di `data/linimasa-regulasi.csv`):

1. **E0 Sebelum PP** (Jan 2022 - 14 Mei 2023): ekspor masih dilarang (Kepmenperindag
   117/2003); pemberitaan didominasi penambangan ilegal dan kasusnya.
2. **E1 PP disahkan** (15 Mei - 31 Des 2023): PP 26/2023 diteken 15 Mei 2023,
   membuka lagi pemanfaatan pasir laut termasuk ekspor. Banjir kritik.
3. **E2 Aturan persiapan** (1 Jan - 28 Agu 2024): Kepmen KP 6/2024 (harga patokan)
   dan Kepmen KP 47/2024 (spesifikasi pasir ekspor); relatif sepi.
4. **E3 Ekspor dibuka** (29 Agu 2024 - 1 Jun 2025): Permendag 20 dan 21/2024
   diundangkan 29 Agustus 2024; keran ekspor resmi dibuka lagi setelah 20 tahun.
5. **E4 Pasca putusan MA** (2 Jun 2025 - 22 Sep 2026): putusan MA 5/P/HUM/2025
   (2 Juni 2025) melarang kembali ekspor; Permen KKP 6/2026 mengukuhkannya.

In [3]:
LABEL = ["menolak", "mendukung", "netral", "campuran"]
EPISODE = ["E0_sebelum_pp", "E1_pp_disahkan", "E2_aturan_persiapan",
           "E3_ekspor_dibuka", "E4_pasca_putusan_ma"]
NAMA = {"E0_sebelum_pp": "E0\nSebelum PP", "E1_pp_disahkan": "E1\nPP disahkan",
        "E2_aturan_persiapan": "E2\nAturan persiapan", "E3_ekspor_dibuka": "E3\nEkspor dibuka",
        "E4_pasca_putusan_ma": "E4\nPasca putusan MA"}

def wilson(k, n):
    lo, hi = proportion_confint(k, n, alpha=0.05, method="wilson")
    p = k / n
    return p, p - lo, hi - p

stat = {}
for ep in EPISODE:
    s = sub[sub["episode"] == ep]
    n = len(s)
    row = {"n": n}
    for lab in ["menolak", "mendukung"]:
        k = int((s["label"] == lab).sum())
        p, lo, hi = wilson(k, n)
        row[f"{lab}_k"] = k
        row[f"{lab}_p"] = p
        row[f"{lab}_lo"] = lo
        row[f"{lab}_hi"] = hi
    stat[ep] = row
tabel = pd.DataFrame(stat).T
print(tabel[["n", "menolak_k", "menolak_p", "menolak_lo", "menolak_hi",
             "mendukung_k", "mendukung_p"]].round(3).to_string())

                         n  menolak_k  menolak_p  menolak_lo  menolak_hi  mendukung_k  mendukung_p
E0_sebelum_pp         34.0        9.0      0.265       0.119       0.166          0.0        0.000
E1_pp_disahkan       259.0       83.0      0.320       0.054       0.059          5.0        0.019
E2_aturan_persiapan   59.0        9.0      0.153       0.070       0.113          2.0        0.034
E3_ekspor_dibuka     248.0       80.0      0.323       0.055       0.061          7.0        0.028
E4_pasca_putusan_ma  135.0       48.0      0.356       0.076       0.084          2.0        0.015


**Alat Analisis: selisih proporsi dengan interval kepercayaan.** Proporsi saja
bisa menipu kalau sampelnya kecil; interval kepercayaan Wilson menunjukkan
rentang yang masuk akal untuk proporsi sejati. Selisih antar-babak diuji
dengan metode skor Newcombe (Wilson): kalau interval selisihnya tidak
menyentuh nol, bedanya sulit dijelaskan sebagai kebetulan sampling saja.

In [4]:
def kontras(ep_a, ep_b):
    a, b = stat[ep_a], stat[ep_b]
    lo, hi = confint_proportions_2indep(
        a["menolak_k"], a["n"], b["menolak_k"], b["n"], method="newcomb")
    return a["menolak_p"] - b["menolak_p"], lo, hi

print("Pembanding utama: babak jeda E2 (aturan persiapan, paling sepi).")
for pasangan in [("E1_pp_disahkan", "E2_aturan_persiapan"),
                 ("E3_ekspor_dibuka", "E2_aturan_persiapan"),
                 ("E4_pasca_putusan_ma", "E2_aturan_persiapan"),
                 ("E1_pp_disahkan", "E0_sebelum_pp")]:
    d, lo, hi = kontras(*pasangan)
    tanda = "signifikan" if lo > 0 else "tidak signifikan"
    print(f"{pasangan[0]} - {pasangan[1]}: selisih {d:+.3f} "
          f"(CI95% {lo:+.3f} s.d. {hi:+.3f}) -> {tanda}")

Pembanding utama: babak jeda E2 (aturan persiapan, paling sepi).
E1_pp_disahkan - E2_aturan_persiapan: selisih +0.168 (CI95% +0.043 s.d. +0.260) -> signifikan
E3_ekspor_dibuka - E2_aturan_persiapan: selisih +0.170 (CI95% +0.045 s.d. +0.263) -> signifikan
E4_pasca_putusan_ma - E2_aturan_persiapan: selisih +0.203 (CI95% +0.067 s.d. +0.312) -> signifikan
E1_pp_disahkan - E0_sebelum_pp: selisih +0.056 (CI95% -0.119 s.d. +0.188) -> tidak signifikan


In [5]:
def grafik_1(mode):
    fig, ax, fs = gaya.dasar(
        mode,
        "Di tiap babak kebijakan, suara menolak selalu paling keras terdengar",
        "Proporsi judul menolak: 26,5% (pra-PP) -> 32,0% (PP) -> 15,3% (jeda) -> 32,3% (ekspor) -> 35,6% (pasca-MA)",
        "Google News RSS, 735 judul relevan, Jan 2022-Sep 2026", len(sub),
    )
    x = np.arange(len(EPISODE))
    p = [stat[e]["menolak_p"] for e in EPISODE]
    err = [[stat[e]["menolak_lo"] for e in EPISODE], [stat[e]["menolak_hi"] for e in EPISODE]]
    puncak = max(p)
    warna = [gaya.AKSEN if abs(v - puncak) < 1e-9 else gaya.INK for v in p]
    ax.bar(x, p, width=0.55, color=warna, zorder=3)
    ax.errorbar(x, p, yerr=err, fmt="none", ecolor=gaya.INK2, elinewidth=1.6, capsize=5, zorder=4)
    for xi, e in zip(x, EPISODE):
        r = stat[e]
        ax.text(xi, r["menolak_p"] + r["menolak_hi"] + 0.012,
                f"{gaya.idn(r['menolak_p'] * 100, 1)}%", ha="center",
                fontsize=8.5 * fs, color=gaya.INK, fontweight="bold")
        ax.text(xi, -0.055, f"n={r['n']}", ha="center", fontsize=8 * fs, color=gaya.INK2)
    ax.set_xticks(x, [NAMA[e] for e in EPISODE], fontsize=8.5 * fs)
    ax.set_ylim(0, 0.48)
    ax.set_ylabel("Proporsi judul bernada menolak", fontsize=9.5 * fs)
    ax.grid(False, axis="x")
    gaya.simpan(fig, "01-suara-per-babak", mode)


for mode in gaya.MODE:
    grafik_1(mode)

In [6]:
BULAN_ID = ["Januari", "Februari", "Maret", "April", "Mei", "Juni", "Juli",
            "Agustus", "September", "Oktober", "November", "Desember"]
bulanan = sub.set_index("tanggal").resample("MS").size()
puncak_bulan = bulanan.idxmax()
nama_puncak = f"{BULAN_ID[puncak_bulan.month - 1]} {puncak_bulan.year}"
TONGGAK = [
    ("2023-05-15", "PP 26/2023"),
    ("2024-08-29", "Permendag 20-21/2024"),
    ("2025-06-02", "Putusan MA"),
    ("2026-03-06", "Permen KKP 6/2026"),
]

def grafik_2(mode):
    fig, ax, fs = gaya.dasar(
        mode,
        "Setiap kali bandul regulasi berayun, pemberitaan ikut melonjak",
        f"Puncak {nama_puncak} ({bulanan.max()} judul), sebulan setelah PP 26/2023 diteken",
        "Google News RSS, 735 judul relevan, Jan 2022-Sep 2026", len(sub),
    )
    ax.bar(bulanan.index, bulanan.values, width=22, color=gaya.INK, zorder=3)
    ax.bar([puncak_bulan], [bulanan.max()], width=22, color=gaya.AKSEN, zorder=4)
    for tgl, nama in TONGGAK:
        t = pd.Timestamp(tgl)
        ax.axvline(t, color=gaya.AKSEN, lw=1.3, ls="--", zorder=2)
        ax.text(t, bulanan.max() * 0.97, nama, rotation=90, ha="right", va="top",
                fontsize=7.5 * fs, color=gaya.AKSEN)
    ax.set_ylabel("Jumlah judul berita per bulan", fontsize=9.5 * fs)
    ax.grid(False, axis="x")
    gaya.simpan(fig, "02-volume-dan-tonggak", mode)


for mode in gaya.MODE:
    grafik_2(mode)

## Temuan

**Benar**: suara yang terdengar lewat judul berita condong menolak, dan condongnya
membesar setiap kali bandul regulasi berayun. Dibanding babak jeda (aturan
persiapan yang sepi), proporsi judul menolak naik 16,8 poin persentase saat PP
disahkan, 17,0 poin saat keran ekspor dibuka, dan 20,3 poin pasca putusan MA;
ketiganya signifikan (CI Newcombe tidak menyentuh nol). Sisi pendukung nyaris
tak terdengar: maksimal 3,4 persen judul per babak.

Dua catatan jujur. Pertama, babak pasca putusan MA justru punya proporsi
menolak tertinggi; judul bernada merayakan pelarangan tetap dihitung sebagai
suara menolak ekspor, sesuai protokol. Kedua, pembanding pra-PP (E0) hanya 34
judul, sehingga selisih E1-E0 (+5,6 poin) tidak signifikan; klaim kenaikan
dipatok pada kontras terhadap babak jeda, bukan terhadap E0.

In [8]:
def kartu_teks():
    gaya.simpan(gaya.kartu("linkedin", "EDISI 004 · LINGKUNGAN · DESEMBER 2026",
        "Pengerukan pasir laut:\nsuara publik dan aturan\nberpihak ke mana?",
        [(gaya.INK2, "Dilarang 2003, dibuka 2023,\ndiekspor lagi 2024, dilarang\nlagi oleh MA pada 2025.\nSaya ukur nada 735 judul\nberita sepanjang ayunan itu."),
         (gaya.AKSEN, "(jawabannya di dalam)")],
        "Jurnal Eksplorasi Data Keseharian"), "slide-01-pertanyaan", "linkedin", penuh=True)

    gaya.simpan(gaya.kartu("linkedin", "EDISI 004 · LINGKUNGAN",
        "Datanya dari mana?",
        [(gaya.INK, "Google News RSS: 1.371 judul,\n5 frasa kunci, Jan 2022-Sep 2026."),
         (gaya.INK2, "735 judul relevan dilabeli\nmenolak/mendukung/netral oleh\nleksikon terbuka, diaudit acak\ndengan kesepakatan 93%.\nSatuan analisisnya judul,\nbukan jajak pendapat.")],
        "Jurnal Eksplorasi Data Keseharian"), "slide-02-data", "linkedin", penuh=True)

    gaya.simpan(gaya.kartu("linkedin", "EDISI 004 · LINGKUNGAN",
        "Temuan dan batasnya",
        [(gaya.AKSEN, "Suara menolak 10-20 kali lebih\nsering terdengar daripada suara\nmendukung, di semua babak."),
         (gaya.INK2, "Batas: judul berita adalah\nnada redaksi, bukan suara warga;\npembanding pra-PP hanya 34 judul."),
         (gaya.INK2, "Sumber: Google News RSS,\npanen 22 Sep 2026.")],
        "Jurnal Eksplorasi Data Keseharian"), "slide-05-batas", "linkedin", penuh=True)


kartu_teks()
print("karousel:", sorted(p.name for p in Path("linkedin/gambar").glob("*.png")))

karousel: ['01-suara-per-babak.png', '02-volume-dan-tonggak.png', 'slide-01-pertanyaan.png', 'slide-02-data.png', 'slide-05-batas.png']
